# ADDS - Quick Start Tutorial

**AI Anticancer Drug Discovery System**  
인하대학병원 의생명공학과

이 노트북은 ADDS의 핵심 기능을 빠르게 체험할 수 있는 튜토리얼입니다.

## 1. Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

print("✓ Setup complete")

## 2. 약물 시너지 계산

Doxorubicin과 Cisplatin의 조합 시너지를 계산합니다.

In [ ]:
from utils.synergy_calculator import SynergyCalculator

calculator = SynergyCalculator()

# 실험 데이터
synergy = calculator.calculate_all_synergies(
    dose_a=10.0,          # Doxorubicin 10 μM
    dose_b=15.0,          # Cisplatin 15 μM
    effect_a=0.35,        # Dox 단독 효과
    effect_b=0.30,        # Cis 단독 효과
    effect_combination=0.75,  # 조합 효과
    ic50_a=5.0,
    ic50_b=8.0
)

print("\n=== Synergy Scores ===")
print(f"Bliss:  {synergy['bliss']:.3f}")
print(f"HSA:    {synergy['hsa']:.3f}")
print(f"Loewe:  {synergy['loewe']:.3f}")
print(f"ZIP:    {synergy['zip']:.3f}")
print(f"\nSynergistic: {synergy['is_synergistic']}")
print(f"Mean Score: {synergy['mean_synergy']:.3f}")

## 3. 시너지 스코어 시각화

In [ ]:
methods = ['bliss', 'hsa', 'loewe', 'zip']
scores = [synergy[m] for m in methods]

plt.figure(figsize=(10, 6))
colors = ['green' if s > 0 else 'red' for s in scores]
plt.bar(methods, scores, color=colors, alpha=0.7, edgecolor='black')
plt.axhline(y=0, color='black', linestyle='--', linewidth=1)
plt.ylabel('Synergy Score', fontsize=12)
plt.title('Doxorubicin + Cisplatin Synergy Analysis', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)

for i, (method, score) in enumerate(zip(methods, scores)):
    plt.text(i, score + 0.01, f'{score:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 4. 데이터 통합

실험 데이터를 구조화하여 저장합니다.

In [ ]:
from data.data_integrator import DataIntegrator

integrator = DataIntegrator()

# 실험 생성
experiment = integrator.create_experiment_record(
    experiment_name="Dox+Cis Combination Study",
    experiment_type="cell_viability",
    cell_line="MCF-7",
    performed_by="연구팀",
    description="항암제 칵테일 시너지 테스트"
)

print(f"✓ Experiment created: {experiment['experiment_id']}")

# 약물 조합 추가
combination = integrator.add_drug_combination(
    experiment_id=experiment['experiment_id'],
    compounds=['Doxorubicin', 'Cisplatin'],
    concentrations=[10.0, 15.0],
    concentration_units='μM'
)

print(f"✓ Combination: {combination['combination_name']}")

# 결과 추가
integrator.add_experimental_result(
    experiment_id=experiment['experiment_id'],
    combination_id=combination['combination_id'],
    result_type='viability',
    value=0.75,
    unit='fraction',
    standard_deviation=0.05,
    n_replicates=3
)

print("✓ Experimental result added")

## 5. 통합 데이터셋 생성

In [ ]:
dataset = integrator.create_integrated_dataset()

print(f"\nDataset Shape: {dataset.shape}")
print("\nDataset Preview:")
dataset.head()

## 6. 데이터 검증

In [ ]:
from evaluation.data_validator import DataQualityValidator

validator = DataQualityValidator()

# 실험 데이터 검증
validation_report = validator.validate_experiment_data(experiment)

print("=== Validation Report ===")
print(f"Valid: {validation_report['valid']}")
if validation_report['issues']:
    print(f"Issues: {validation_report['issues']}")
else:
    print("No issues found!")

## 7. 내보내기

In [ ]:
# JSON으로 내보내기
output_path = Path('../data/outputs/tutorial_data.json')
output_path.parent.mkdir(parents=True, exist_ok=True)

integrator.export_to_json(output_path)

print(f"✓ Data exported to: {output_path}")

## Summary

이 튜토리얼에서 다룬 내용:

✅ 약물 시너지 계산 (Bliss, Loewe, HSA, ZIP)  
✅ 시너지 스코어 시각화  
✅ 실험 데이터 구조화 및 통합  
✅ 데이터 품질 검증  
✅ JSON 형식으로 내보내기

더 많은 기능은 [README.md](../README.md)를 참고하세요!